# Hoan tat TAT CA checkpoint Track A bi Kaggle timeout tu du lieu co san

Dung cho MOI model (geoformerdock, gnina_dense, gnina_default2018, pafnucy) —
tu dong do TAT CA thu muc dang `<model>_valsplit_s2026/best_model.pt` trong
`/kaggle/input/` (bat ke nam trong Dataset nao), copy vao dung vi tri, roi
chay `tools/finalize_from_checkpoint.py` cho TUNG checkpoint tim duoc — khong
train lai, chi chon nguong + danh gia + ghi summary.json (vai phut/model).

**TRUOC KHI CHAY**:
1. Settings -> Accelerator -> GPU T4 x2, Internet -> On.
2. Add Data -> gan dataset chua `data/` (vd `geoformerdock-output`) VA cac
   dataset chua checkpoint moi (vd chua `gnina_dense_valsplit_s2026/best_model.pt`,
   `pafnucy_valsplit_s2026/best_model.pt`) — co the gan nhieu dataset cung luc,
   notebook tu dong tim trong tat ca.

**Mac dinh dung batch_size=256 cho moi model khi finalize** (xac nhan qua log:
khong co dong OOM/THAT BAI o batch_size=256 truoc du lieu epoch that trong ca
gnina_dense va pafnucy — 2 model deu chay thang duoc voi 256 tu dau). Neu 1
model cu the that su can batch nho hon, sua bien BATCH_SIZE_OVERRIDES ben duoi
truoc khi chay.

## 0. Cai dat thu vien truoc tien (khong can restart kernel)

In [ ]:
!pip install -q 'numpy<2' molgrid mlflow
!pip install -q --no-deps pytorch-ignite


In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, '-c', '''
import numpy, torch, molgrid
print("numpy:", numpy.__version__)
print("torch:", torch.__version__)
print("torch.cuda.is_available():", torch.cuda.is_available())
print("molgrid: import OK")
assert numpy.__version__.startswith("1."), f"numpy={numpy.__version__} van >=2"
assert torch.cuda.is_available(), (
    f"torch.__version__={torch.__version__} KHONG thay GPU (co the pip install da "
    "vo tinh keo torch ve ban CPU-only, hoac Settings Accelerator chua bat GPU). "
    "Kiem tra Settings panel phai, hoac bao Claude kem dong nay."
)
print("OK")
'''], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr, file=sys.stderr)
    raise RuntimeError('Kiem tra thu vien THAT BAI - dan cho Claude.')


## 1. Lay code moi nhat tu GitHub

In [ ]:
import os
if os.path.isdir('/kaggle/working/VNICT2026_Docking_Paper'):
    !cd /kaggle/working/VNICT2026_Docking_Paper && git pull
else:
    !cd /kaggle/working && git clone https://github.com/ducnm-mimhus/VNICT2026_Docking_Paper.git


In [ ]:
%cd /kaggle/working/VNICT2026_Docking_Paper
!git log --oneline -1
!test -f tools/finalize_from_checkpoint.py && echo 'OK: tools/finalize_from_checkpoint.py co san' || echo 'LOI: khong thay tools/finalize_from_checkpoint.py'


## 2. Tim data/ + TAT CA checkpoint `*_valsplit_s2026/best_model.pt` trong /kaggle/input

In [ ]:
import glob, os, re, subprocess

print('=== /kaggle/input ===')
print(subprocess.run(['ls', '-la', '/kaggle/input'], capture_output=True, text=True).stdout)

data_candidates = [
    p for p in glob.glob('/kaggle/input/**/data', recursive=True)
    if os.path.isdir(p) and os.path.isdir(os.path.join(p, 'types'))
]
assert len(data_candidates) >= 1, 'Khong tim thay data/ (co types/) trong /kaggle/input.'
SRC_DATA = data_candidates[0]
print(f'Dung data/ tu: {SRC_DATA}')

ckpt_candidates = glob.glob('/kaggle/input/**/*_valsplit_s2026/best_model.pt', recursive=True)
assert len(ckpt_candidates) >= 1, 'Khong tim thay checkpoint *_valsplit_s2026/best_model.pt nao trong /kaggle/input.'

FOUND = {}  # model_name -> src checkpoint path
for p in ckpt_candidates:
    dirname = os.path.basename(os.path.dirname(p))  # vd 'gnina_dense_valsplit_s2026'
    m = re.match(r'^(.+)_valsplit_s2026$', dirname)
    if not m:
        continue
    model_name = m.group(1)
    FOUND.setdefault(model_name, p)  # neu trung, giu cai dau tien tim thay

print(f'\nTim thay {len(FOUND)} checkpoint:')
for model_name, p in FOUND.items():
    print(f'  {model_name}: {p}')


In [ ]:
import shutil, os

DST_DATA = '/kaggle/working/VNICT2026_Docking_Paper/data'
if os.path.isdir(DST_DATA):
    print(f'{DST_DATA} da ton tai - bo qua copy.')
else:
    print(f'Dang copy {SRC_DATA} -> {DST_DATA} (noi bo, khong qua internet, ~5.5GB, vai phut)...')
    shutil.copytree(SRC_DATA, DST_DATA)
    print('Xong copy data/.')

CKPT_DST = {}
for model_name, src in FOUND.items():
    dst_dir = f'/kaggle/working/VNICT2026_Docking_Paper/results/models/{model_name}_valsplit_s2026'
    os.makedirs(dst_dir, exist_ok=True)
    dst = os.path.join(dst_dir, 'best_model.pt')
    shutil.copy2(src, dst)
    CKPT_DST[model_name] = dst
    print(f'{model_name}: da copy vao {dst}')


## 3. Chay finalize cho TUNG checkpoint tim duoc

Khong train lai — chi fit target_normalizer tren train + eval tren val/test +
quet nguong tren val, cho MOI model, tuan tu.

In [ ]:
BATCH_SIZE_OVERRIDES = {}  # vd {'pafnucy': 128} neu can — mac dinh 256 cho tat ca

for model_name, ckpt_path in CKPT_DST.items():
    bs = BATCH_SIZE_OVERRIDES.get(model_name, 256)
    out_dir = os.path.dirname(ckpt_path)
    print(f'\n########## FINALIZE {model_name} (batch_size={bs}) ##########')
    !python3 tools/finalize_from_checkpoint.py \
        --model {model_name} --batch_size {bs} \
        --checkpoint {ckpt_path} \
        --out_dir {out_dir}


## 4. Ket qua — dan phan nay vao chat cho Claude

In [ ]:
import json
for model_name, ckpt_path in CKPT_DST.items():
    summary_path = os.path.join(os.path.dirname(ckpt_path), 'summary.json')
    print(f'\n===== {model_name} =====')
    if os.path.exists(summary_path):
        print(json.dumps(json.load(open(summary_path)), indent=2, ensure_ascii=False))
    else:
        print('KHONG co summary.json - finalize that bai, xem output cell tren.')
